# Mục 3e — Mã danh mục và thứ tự thành số nguyên
**Võ Minh Huy — 2311110117**

Giữ ý nghĩa mã 0/1; chuyển A1–A4 thành mã thứ tự 1–4. age_group là thuộc tính phát sinh từ tuổi sau N02, không phải cột gốc. Giữ nhãn chưa xác minh ở trạng thái thiếu.

## 1. Chuyển kiểu và lập bảng quy ước
Python dùng kiểu Int64 cho số nguyên có thể chứa giá trị thiếu, tương ứng int NULL trong SQL. Kiểm tra mã hợp lệ trước chuyển, không cắt phần lẻ để biến mã sai thành mã đúng.

In [1]:
from pathlib import Path
from decimal import Decimal
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_vao = thu_muc / "ket_qua/15_khong_nhat_quan/02_death_event/ban_lam_viec_nhan_python.csv"
ban_day_du = pd.read_csv(file_vao, dtype=str, keep_default_na=False)
cot_danh_muc = ["anaemia", "diabetes", "high_blood_pressure", "sex", "smoking", "DEATH_EVENT"]
bang_ma = pd.DataFrame({"MaHoSo":ban_day_du["MaHoSo"].astype(int)})
for cot in cot_danh_muc:
    gia_tri = ban_day_du[cot].map(lambda v: Decimal(v) if v.strip() else None)
    if (~gia_tri.dropna().isin([0, 1])).any():
        raise ValueError("Mã ngoài miền 0/1 ở " + cot)
    if cot != "DEATH_EVENT" and gia_tri.isna().any():
        raise ValueError("Còn mã thiếu ở " + cot)
    bang_ma[cot] = pd.array(gia_tri.tolist(), dtype="Int64")

quy_doi = {"A1":1, "A2":2, "A3":3, "A4":4}
if not ban_day_du["age_group"].isin(quy_doi).all():
    raise ValueError("Nhóm tuổi ngoài A1–A4.")
bang_ma["MaNhomTuoi"] = ban_day_du["age_group"].map(quy_doi).astype("Int64")
quy_uoc = pd.DataFrame({"NhomTuoi":["A1", "A2", "A3", "A4"],
    "KhoangTuoi":["Dưới 50", "50 đến dưới 60", "60 đến dưới 70", "Từ 70 trở lên"],
    "MaNhomTuoi":[1, 2, 3, 4]})
quy_uoc["SoHoSo"] = quy_uoc["MaNhomTuoi"].map(bang_ma["MaNhomTuoi"].value_counts()).fillna(0).astype(int)
quy_uoc

,NhomTuoi,KhoangTuoi,MaNhomTuoi,SoHoSo
0,A1,Dưới 50,1,42
1,A2,50 đến dưới 60,2,77
2,A3,60 đến dưới 70,3,111
3,A4,Từ 70 trở lên,4,69


## 2. Kiểm tra trên toàn bộ hồ sơ
So mã nhóm với khoảng tuổi tính từ age. Mã thứ tự giữ quan hệ tăng dần nhưng không có khoảng cách tuổi bằng nhau. Một nhãn thiếu vẫn giữ thiếu.

In [2]:
def ma_tu_tuoi(v):
    tuoi = Decimal(v)
    if not tuoi.is_finite() or tuoi <= 0:
        raise ValueError("Tuổi không hợp lệ.")
    return 1 if tuoi < 50 else 2 if tuoi < 60 else 3 if tuoi < 70 else 4

ma_kiem_tra = ban_day_du["age"].map(ma_tu_tuoi)
cac_cot_ma = cot_danh_muc + ["MaNhomTuoi"]
tong_ket = pd.DataFrame({"SoHoSo":[len(bang_ma)],
    "SoCotMaKieuInt":[sum(str(bang_ma[c].dtype) == "Int64" for c in cac_cot_ma)],
    "SoNhomTuoi":[bang_ma["MaNhomTuoi"].nunique()],
    "SoSaiKhoangTuoi":[bang_ma["MaNhomTuoi"].ne(ma_kiem_tra).sum()],
    "SoNhanChoXacMinh":[bang_ma["DEATH_EVENT"].isna().sum()]})
tong_ket

,SoHoSo,SoCotMaKieuInt,SoNhomTuoi,SoSaiKhoangTuoi,SoNhanChoXacMinh
0,299,7,4,0,1


## 3. Một ví dụ mỗi nhóm tuổi
MaHoSo là số thứ tự nội bộ, có thể khác SQL. Bốn ví dụ chỉ minh họa; kiểm tra toàn bộ đã thực hiện ở mục 2.

In [3]:
vi_du = pd.DataFrame({"MaHoSo":bang_ma["MaHoSo"],
    "age":ban_day_du["age"].map(Decimal), "NhomTuoiTruoc":ban_day_du["age_group"],
    "MaThuTuSau":bang_ma["MaNhomTuoi"]})
vi_du = (vi_du.sort_values(["MaThuTuSau", "MaHoSo"])
    .groupby("MaThuTuSau", sort=False).head(1).reset_index(drop=True))
vi_du

,MaHoSo,age,NhomTuoiTruoc,MaThuTuSau
0,13,45.000000,A1,1
1,2,55.000000,A2,2
2,3,65.000000,A3,3
3,1,75.000000,A4,4


**Nhận xét:** 299 hồ sơ, 7 cột mã số nguyên, 4 nhóm tuổi, 0 sai khoảng tuổi, 1 nhãn chờ xác minh. Số hồ sơ các nhóm lần lượt 42, 77, 111, 69. Không tạo ngày giả hoặc chuỗi danh sách giả; giữ dữ liệu đầy đủ và nhãn thiếu.

## 4. Lưu bảng mã riêng
Đây là bảng các mã, không phải dữ liệu after đầy đủ.

In [4]:
thu_muc_kq = thu_muc / "ket_qua/17_chuyen_doi/02_danh_muc_thu_tu"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
quy_uoc.to_csv(thu_muc_kq / "quy_uoc_python.csv", index=False, encoding="utf-8-sig")
tong_ket.to_csv(thu_muc_kq / "tong_ket_python.csv", index=False, encoding="utf-8-sig")
bang_ma.to_csv(thu_muc_kq / "ma_danh_muc_thu_tu_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu bảng mã danh mục và thứ tự; giữ nguyên nguồn."]})

,TrangThai
0,Đã lưu bảng mã danh mục và thứ tự; giữ nguyên ...
